# PEGASUS Mitigation — Same Hallucination Calculation as Original Code

This notebook keeps the **hallucination calculation the same as your original PEGASUS notebook**.

Old-style hallucination rule:

1. Split the article into chunks.
2. Check every summary sentence against every chunk using NLI.
3. If **any chunk gives contradiction**, the sentence/summary is hallucinated.
4. If **no chunk gives entailment**, the sentence/summary is hallucinated.
5. A summary is marked hallucinated if **any sentence** is hallucinated.

This is stricter than the newer entailment-priority logic. The original and mitigated PEGASUS summaries are evaluated using the **same strict old-style function**, so the comparison is fair.

Mitigation in this notebook does **not delete sentences**. If a neutral/contradictory sentence cannot be fixed, it is kept and counted as unresolved hallucination.

In [ ]:
# Install required packages
!pip uninstall -y transformers sentencepiece -q
!pip install transformers==4.41.2 sentencepiece -q
!pip install evaluate rouge_score bert-score nltk openpyxl -q
print('All packages installed')

In [ ]:
# Optional: clear PEGASUS cache if you were facing model-loading issues
!rm -rf ~/.cache/huggingface/hub/models--google--pegasus-cnn_dailymail

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import evaluate
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.tokenize import sent_tokenize
from transformers import PegasusTokenizer, PegasusForConditionalGeneration, pipeline
from bert_score import score as bert_score
from tqdm import tqdm

# Label cache — keyed by (article_text_hash, sentence) to avoid stale hits
# on partial reruns or dataset switches.
import hashlib

def _article_hash(article):
    """Short hash of article text used as cache key instead of a fragile integer index."""
    return hashlib.md5(str(article).encode()).hexdigest()[:12]

_label_cache = {}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

In [ ]:
# Load PEGASUS summarization model
model_name = 'google/pegasus-cnn_dailymail'

pegasus_tokenizer = PegasusTokenizer.from_pretrained(model_name)
pegasus_model = PegasusForConditionalGeneration.from_pretrained(model_name).to(device)
pegasus_model.eval()

print('PEGASUS loaded')

In [ ]:
# Load NLI model used for hallucination detection
# Labels: entailment, neutral, contradiction
nli_model = pipeline(
    'text-classification',
    model='facebook/bart-large-mnli',
    device=0 if torch.cuda.is_available() else -1
)

print('NLI model loaded')

## Load dataset

Change `DATA_FILE` if your file name is different.

Supported formats:
- `.csv`
- `.parquet`
- `.json`

The code tries to automatically detect article and reference-summary columns.

In [ ]:
# Change this file name according to your dataset
# Examples:
# DATA_FILE = 'test.csv'
# DATA_FILE = 'test-premed.parquet'
# DATA_FILE = 'test.json'
DATA_FILE = 'test.csv'

if DATA_FILE.endswith('.csv'):
    test_df = pd.read_csv(DATA_FILE)
elif DATA_FILE.endswith('.parquet'):
    test_df = pd.read_parquet(DATA_FILE)
elif DATA_FILE.endswith('.json'):
    test_df = pd.read_json(DATA_FILE)
else:
    raise ValueError('Unsupported file type. Use CSV, Parquet, or JSON.')

print(test_df.head())
print(test_df.columns)
print('Total samples:', len(test_df))

# Automatically detect article column
article_candidates = ['article', 'document', 'text']
summary_candidates = ['highlights', 'summary', 'abstract', 'reference_summary']

article_col = next((c for c in article_candidates if c in test_df.columns), None)
summary_col = next((c for c in summary_candidates if c in test_df.columns), None)

if article_col is None:
    raise ValueError(f'Could not find article column. Available columns: {list(test_df.columns)}')
if summary_col is None:
    raise ValueError(f'Could not find reference summary column. Available columns: {list(test_df.columns)}')

print('Using article column:', article_col)
print('Using reference summary column:', summary_col)

SAMPLE_SIZE = 10
RANDOM_STATE = 26

df_sample = test_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

articles = df_sample[article_col].astype(str).tolist()
reference_summaries = df_sample[summary_col].astype(str).tolist()

print('Articles:', len(articles))
print('Reference summaries:', len(reference_summaries))

## Old-style NLI hallucination functions

This section is the most important part.

The function below uses the same logic as your original PEGASUS notebook:

```python
if "contradiction" in labels:
    hallucinated
if "entailment" not in labels:
    hallucinated
```

That means contradiction has priority over entailment, exactly like your old code.

In [ ]:
def get_chunks(article, chunk_size=400, overlap=50):
    """Split article into overlapping word chunks."""
    words = str(article).split()
    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
        # BUG FIX: removed premature break — range() already terminates
        # at the end of the word list, so the break was causing the last
        # chunk to be skipped when boundaries aligned exactly.

    return chunks if chunks else ['']


def get_nli_label(premise, hypothesis):
    """Get NLI label for one chunk and one summary sentence."""
    result = nli_model(
        {'text': premise, 'text_pair': hypothesis},
        truncation=True
    )
    # BUG FIX: pipeline may return a list (top-k results) depending on
    # transformers version and top_k setting — always take the first item.
    if isinstance(result, list):
        result = result[0]
    return result['label'].lower()


def get_labels_across_chunks(chunks, sentence):
    """Return all NLI labels for one sentence across all article chunks."""
    return [get_nli_label(chunk, sentence) for chunk in chunks]


def old_style_sentence_label_from_labels(labels):
    """
    Convert many chunk labels into one final sentence label using OLD PEGASUS logic.

    OLD LOGIC:
    - If any chunk says contradiction -> contradiction
    - Else if at least one chunk says entailment -> entailment
    - Else -> neutral
    """
    if 'contradiction' in labels:
        return 'contradiction'
    elif 'entailment' in labels:
        return 'entailment'
    else:
        return 'neutral'


def cached_old_style_label(article_id, chunks, sentence):
    """Cached old-style label for speed.

    BUG FIX: cache key now uses a hash of the article text instead of
    a bare integer index.  Integer indices collide if you run the notebook
    on different datasets or subsets without restarting the kernel.
    """
    # chunks[0] is a deterministic fingerprint of the article split
    article_key = _article_hash(chunks[0] if chunks else '')
    key = (article_key, sentence)
    if key not in _label_cache:
        labels = get_labels_across_chunks(chunks, sentence)
        _label_cache[key] = old_style_sentence_label_from_labels(labels)
    return _label_cache[key]


def old_style_sentence_is_hallucinated(article_id, chunks, sentence):
    """
    Same hallucination decision as the original PEGASUS code.
    Neutral and contradiction are hallucination.
    Entailment is not hallucination.
    """
    label = cached_old_style_label(article_id, chunks, sentence)
    return label in ('neutral', 'contradiction')


def old_style_summary_hallucinated(article_id, article, summary):
    """
    Summary-level hallucination score using the OLD PEGASUS logic.
    Returns:
    - 1 if any sentence is hallucinated
    - 0 if no sentence is hallucinated
    """
    sentences = sent_tokenize(str(summary))
    if len(sentences) == 0:
        return 0

    chunks = get_chunks(article)

    for sent in sentences:
        if old_style_sentence_is_hallucinated(article_id, chunks, sent):
            return 1

    return 0


def old_style_sentence_hallucination_rate(article_id, article, summary):
    """
    Optional sentence-level strict hallucination rate.
    This is NOT the final old PEGASUS score, but useful for debugging.
    """
    sentences = sent_tokenize(str(summary))
    if len(sentences) == 0:
        return 0.0

    chunks = get_chunks(article)
    hallucinated = sum(
        1 for sent in sentences
        if old_style_sentence_is_hallucinated(article_id, chunks, sent)
    )
    return hallucinated / len(sentences)


def faithfulness_score_old_style(article_id, article, summary):
    """
    Used only for choosing the best candidate summary.
    Higher score means more entailed sentences under the same old-style label logic.
    """
    sentences = sent_tokenize(str(summary))
    if len(sentences) == 0:
        return 0.0

    chunks = get_chunks(article)
    entailed = sum(
        1 for sent in sentences
        if cached_old_style_label(article_id, chunks, sent) == 'entailment'
    )
    return entailed / len(sentences)

print('Old-style hallucination functions defined')

## Mitigation functions

This mitigation does not delete unsupported sentences.

For every selected summary:

1. Keep entailed sentences.
2. For neutral or contradiction sentences, find the best evidence sentence from the source article.
3. Replace only if the evidence sentence is **entailed under the same old-style logic**.
4. If the evidence sentence is still neutral/contradiction, keep the original sentence and count it as unresolved.

This prevents the hallucination rate from becoming 0 just because sentences were removed.

In [ ]:
def get_best_evidence_chunk(article, sentence):
    """Find the article chunk with highest word overlap with the problematic sentence."""
    chunks = get_chunks(article)
    sent_words = set(str(sentence).lower().split())

    best_chunk = chunks[0]
    best_overlap = -1

    for chunk in chunks:
        overlap = len(sent_words & set(chunk.lower().split()))
        if overlap > best_overlap:
            best_overlap = overlap
            best_chunk = chunk

    return best_chunk


def select_best_evidence_sentence(evidence_chunk, original_sentence):
    """Select the most related real sentence from the evidence chunk."""
    candidate_sentences = sent_tokenize(str(evidence_chunk))
    if not candidate_sentences:
        return original_sentence

    orig_words = set(str(original_sentence).lower().split())
    best_sent = candidate_sentences[0]
    best_overlap = -1

    for cand in candidate_sentences:
        overlap = len(orig_words & set(cand.lower().split()))
        if overlap > best_overlap:
            best_overlap = overlap
            best_sent = cand

    return best_sent.strip()


def mitigate_summary_old_style(article_id, article, candidates):
    """
    Select best candidate, then try evidence-sentence substitution.
    Uses the SAME old-style hallucination logic for label decisions.
    Does NOT delete sentences.
    """
    # Select best candidate using old-style faithfulness score
    scored = [(faithfulness_score_old_style(article_id, article, c), c) for c in candidates]
    best_score, best_candidate = max(scored, key=lambda x: x[0])

    sentences = sent_tokenize(str(best_candidate))
    chunks = get_chunks(article)

    final_sentences = []

    n_entailed = 0
    n_neutral_fixed = 0
    n_neutral_unresolved = 0
    n_contra_fixed = 0
    n_contra_unresolved = 0

    for sent in sentences:
        label = cached_old_style_label(article_id, chunks, sent)

        if label == 'entailment':
            final_sentences.append(sent)
            n_entailed += 1

        elif label == 'neutral':
            evidence_chunk = get_best_evidence_chunk(article, sent)
            substitute = select_best_evidence_sentence(evidence_chunk, sent)
            sub_label = cached_old_style_label(article_id, chunks, substitute)

            if sub_label == 'entailment':
                final_sentences.append(substitute)
                n_neutral_fixed += 1
            else:
                # Do not remove. Keep original and count unresolved.
                final_sentences.append(sent)
                n_neutral_unresolved += 1

        else:  # contradiction
            evidence_chunk = get_best_evidence_chunk(article, sent)
            substitute = select_best_evidence_sentence(evidence_chunk, sent)
            sub_label = cached_old_style_label(article_id, chunks, substitute)

            if sub_label == 'entailment':
                final_sentences.append(substitute)
                n_contra_fixed += 1
            else:
                # Do not remove. Keep original and count unresolved.
                final_sentences.append(sent)
                n_contra_unresolved += 1

    mitigated = ' '.join(final_sentences)

    stats = {
        'total_sentences': len(sentences),
        'n_entailed': n_entailed,
        'n_neutral_fixed': n_neutral_fixed,
        'n_neutral_unresolved': n_neutral_unresolved,
        'n_contra_fixed': n_contra_fixed,
        'n_contra_unresolved': n_contra_unresolved,
    }

    return mitigated, stats

print('Mitigation functions defined')

## Generate PEGASUS summaries

PEGASUS generates 4 candidate summaries using beam search.

- `pegasus_original_summaries` = first beam output, comparable to your old one-summary PEGASUS output.
- `pegasus_mitigated_summaries` = best candidate by old-style faithfulness + evidence substitution.

In [ ]:
# Clear label cache before each run to avoid stale results from prior datasets
_label_cache.clear()

pegasus_original_summaries = []
pegasus_mitigated_summaries = []
pegasus_stats_list = []
all_candidate_summaries = []

for article_id, article in enumerate(tqdm(articles, desc='PEGASUS')):
    inputs = pegasus_tokenizer(
        str(article),
        max_length=1024,
        truncation=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        summary_ids = pegasus_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=128,
            min_length=32,
            num_beams=8,
            num_return_sequences=4,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            early_stopping=True
        )

    candidates = [
        pegasus_tokenizer.decode(
            s,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )
        for s in summary_ids
    ]

    all_candidate_summaries.append(candidates)

    # Original = first beam, same idea as old PEGASUS code returning summary_ids[0]
    pegasus_original_summaries.append(candidates[0])

    # Mitigated = best candidate + evidence substitution, using old-style labels
    mitigated, stats = mitigate_summary_old_style(article_id, article, candidates)
    pegasus_mitigated_summaries.append(mitigated)
    pegasus_stats_list.append(stats)

print('Completed!')
print('Original summaries:', len(pegasus_original_summaries))
print('Mitigated summaries:', len(pegasus_mitigated_summaries))

In [ ]:
# Show first few examples
for i in range(min(3, len(articles))):
    print(f'ARTICLE {i+1}')
    print('
REFERENCE SUMMARY:')
    print(reference_summaries[i])
    print('
PEGASUS ORIGINAL SUMMARY:')
    print(pegasus_original_summaries[i])
    print('
PEGASUS MITIGATED SUMMARY:')
    print(pegasus_mitigated_summaries[i])
    print('
' + '='*100 + '
')

## Hallucination evaluation using the same old-style calculation

This is the fair part:

- Original PEGASUS is checked using old-style logic.
- Mitigated PEGASUS is checked using the same old-style logic.

So if the score changes, it is because the summary changed, not because the evaluation method changed.

In [ ]:
# Summary-level hallucination score: same style as original PEGASUS notebook
pegasus_hall_orig = [
    old_style_summary_hallucinated(i, article, summary)
    for i, (article, summary) in enumerate(tqdm(
        list(zip(articles, pegasus_original_summaries)),
        desc='Old-style hall. original'
    ))
]

pegasus_hall_mit = [
    old_style_summary_hallucinated(i, article, summary)
    for i, (article, summary) in enumerate(tqdm(
        list(zip(articles, pegasus_mitigated_summaries)),
        desc='Old-style hall. mitigated'
    ))
]

pegasus_hallucination_rate_orig = sum(pegasus_hall_orig) / len(pegasus_hall_orig)
pegasus_hallucination_rate_mit = sum(pegasus_hall_mit) / len(pegasus_hall_mit)

print('PEGASUS Original Hallucinated Summaries:', sum(pegasus_hall_orig))
print('PEGASUS Original Hallucination Rate:', pegasus_hallucination_rate_orig)
print('
PEGASUS Mitigated Hallucinated Summaries:', sum(pegasus_hall_mit))
print('PEGASUS Mitigated Hallucination Rate:', pegasus_hallucination_rate_mit)

In [ ]:
# Optional debugging: sentence-level old-style hallucination rate
pegasus_sentence_hall_orig = [
    old_style_sentence_hallucination_rate(i, article, summary)
    for i, (article, summary) in enumerate(zip(articles, pegasus_original_summaries))
]

pegasus_sentence_hall_mit = [
    old_style_sentence_hallucination_rate(i, article, summary)
    for i, (article, summary) in enumerate(zip(articles, pegasus_mitigated_summaries))
]

print('Average sentence-level hallucination, original:', np.mean(pegasus_sentence_hall_orig))
print('Average sentence-level hallucination, mitigated:', np.mean(pegasus_sentence_hall_mit))

## Mitigation statistics

These stats show whether mitigation changed sentences and whether anything was left unresolved.

No sentences are deleted in this notebook.

In [ ]:
def summarise_stats(stats_list):
    total = sum(s['total_sentences'] for s in stats_list)
    entailed = sum(s['n_entailed'] for s in stats_list)
    neutral_fixed = sum(s['n_neutral_fixed'] for s in stats_list)
    neutral_unresolved = sum(s['n_neutral_unresolved'] for s in stats_list)
    contra_fixed = sum(s['n_contra_fixed'] for s in stats_list)
    contra_unresolved = sum(s['n_contra_unresolved'] for s in stats_list)

    print('--- PEGASUS Mitigation Stats ---')
    print('Total sentences:', total)
    print(f'Entailed kept unchanged        : {entailed} ({entailed/total*100:.1f}%)')
    print(f'Neutral fixed                  : {neutral_fixed} ({neutral_fixed/total*100:.1f}%)')
    print(f'Neutral unresolved kept        : {neutral_unresolved} ({neutral_unresolved/total*100:.1f}%)')
    print(f'Contradiction fixed            : {contra_fixed} ({contra_fixed/total*100:.1f}%)')
    print(f'Contradiction unresolved kept  : {contra_unresolved} ({contra_unresolved/total*100:.1f}%)')
    print('Removed sentences              : 0')

summarise_stats(pegasus_stats_list)

## ROUGE and BERTScore

In [ ]:
rouge = evaluate.load('rouge')

pegasus_rouge_orig = rouge.compute(
    predictions=pegasus_original_summaries,
    references=reference_summaries
)

pegasus_rouge_mit = rouge.compute(
    predictions=pegasus_mitigated_summaries,
    references=reference_summaries
)

print('PEGASUS ROUGE original:', {k: round(v, 4) for k, v in pegasus_rouge_orig.items()})
print('PEGASUS ROUGE mitigated:', {k: round(v, 4) for k, v in pegasus_rouge_mit.items()})

In [ ]:
_, _, F1_peg_orig = bert_score(
    pegasus_original_summaries,
    reference_summaries,
    lang='en',
    verbose=False
)

_, _, F1_peg_mit = bert_score(
    pegasus_mitigated_summaries,
    reference_summaries,
    lang='en',
    verbose=False
)

print('PEGASUS BERTScore original:', round(F1_peg_orig.mean().item(), 4))
print('PEGASUS BERTScore mitigated:', round(F1_peg_mit.mean().item(), 4))

## Final results table

The hallucination score here is the **same old-style summary-level hallucination score** used in your original PEGASUS notebook.

In [ ]:
final_results = pd.DataFrame({
    'Model': ['PEGASUS Original', 'PEGASUS Mitigated'],
    'ROUGE-1': [pegasus_rouge_orig['rouge1'], pegasus_rouge_mit['rouge1']],
    'ROUGE-2': [pegasus_rouge_orig['rouge2'], pegasus_rouge_mit['rouge2']],
    'ROUGE-L': [pegasus_rouge_orig['rougeL'], pegasus_rouge_mit['rougeL']],
    'BERTScore F1': [
        round(F1_peg_orig.mean().item(), 4),
        round(F1_peg_mit.mean().item(), 4)
    ],
    'Old-Style Summary Hall. Score': [
        round(pegasus_hallucination_rate_orig, 4),
        round(pegasus_hallucination_rate_mit, 4)
    ],
    'Hallucinated Summaries': [sum(pegasus_hall_orig), sum(pegasus_hall_mit)],
    'Total Summaries': [len(pegasus_hall_orig), len(pegasus_hall_mit)],
    'Neutral Fixed': ['-', sum(s['n_neutral_fixed'] for s in pegasus_stats_list)],
    'Neutral Unresolved Kept': ['-', sum(s['n_neutral_unresolved'] for s in pegasus_stats_list)],
    'Contradiction Fixed': ['-', sum(s['n_contra_fixed'] for s in pegasus_stats_list)],
    'Contradiction Unresolved Kept': ['-', sum(s['n_contra_unresolved'] for s in pegasus_stats_list)],
    'Removed Sentences': ['-', 0]
})

print(final_results.to_string(index=False))
final_results

In [ ]:
output_df = pd.DataFrame({
    'Article': articles,
    'Reference Summary': reference_summaries,
    'PEGASUS Original': pegasus_original_summaries,
    'PEGASUS Mitigated': pegasus_mitigated_summaries,
    'Original Old-Style Summary Hallucinated': pegasus_hall_orig,
    'Mitigated Old-Style Summary Hallucinated': pegasus_hall_mit,
    'Original Sentence Hall. Rate': pegasus_sentence_hall_orig,
    'Mitigated Sentence Hall. Rate': pegasus_sentence_hall_mit,
})

output_df.to_csv('pegasus_same_hallucination_logic_results.csv', index=False)
output_df.to_excel('pegasus_same_hallucination_logic_results.xlsx', index=False)
final_results.to_csv('pegasus_same_hallucination_logic_final_table.csv', index=False)

print('Saved: pegasus_same_hallucination_logic_results.csv')
print('Saved: pegasus_same_hallucination_logic_results.xlsx')
print('Saved: pegasus_same_hallucination_logic_final_table.csv')

# Uncomment in Google Colab if you want downloads
# from google.colab import files
# files.download('pegasus_same_hallucination_logic_results.csv')
# files.download('pegasus_same_hallucination_logic_results.xlsx')
# files.download('pegasus_same_hallucination_logic_final_table.csv')